# Experimento 02 — Distancia geográfica cliente↔vendedor

Este cuaderno extiende el baseline del experimento 01 con una señal nueva: la distancia física entre el cliente y cada vendedor del pedido, derivada de `olist_geolocation_dataset`. La hipótesis es directa: en un país con la extensión de Brasil, la distancia debería explicar una parte estructural del tiempo de entrega.

El pipeline y la configuración son los mismos del baseline (partición 60/20/20 cronológica, `TimeSeriesSplit` con tres pliegues, bootstrap para IC95%, cinco familias de modelos más la línea base trivial). Lo único que cambia es el conjunto de predictores. Los resultados se guardan en `outputs/experimentos/02_distancia_geografica/`.


## 1. Entorno y datos

Resolvemos la raíz del repo para que los imports de `src` funcionen tanto en local como en Colab, y descargamos los CSV de Olist desde la API pública de Kaggle si todavía no están en `data/raw/`.


In [1]:
from pathlib import Path
import os
import subprocess
import sys


In [2]:
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Cde571/LogisTech-Modelos-Simulacion-II.git'

if IN_COLAB:
    root = Path('/content/LogisTech-Modelos-Simulacion-II')
    if not root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(root)], check=True)
    else:
        subprocess.run(['git', '-C', str(root), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)
    os.chdir(root)
else:
    root = Path.cwd()
    while root.name in ('experimentos', 'notebook', 'notebooks'):
        root = root.parent
    os.chdir(root)

print('Raíz del repositorio:', root)


Raíz del repositorio: /home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II


In [3]:
import urllib.request
import zipfile


In [4]:
raw = root / 'data' / 'raw'
required = raw / 'olist_orders_dataset.csv'

if not required.exists():
    raw.mkdir(parents=True, exist_ok=True)
    archive = root / 'data' / 'olist.zip'
    urllib.request.urlretrieve(
        'https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce',
        archive,
    )
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(raw)

print('CSV disponibles en data/raw:', len(list(raw.glob('*.csv'))))


CSV disponibles en data/raw: 9


## 2. Configuración y dataset enriquecido

Arrancamos con la misma configuración del baseline y construimos el dataset a nivel de pedido. La novedad es `attach_order_geography`, que agrega la coordenada mediana del cliente y la distancia promedio y máxima a los vendedores del pedido. La agregación es segura: la señal está disponible en el instante de la compra y no introduce fuga.


In [5]:
from src.models import ExperimentConfig
from src.preprocessing import build_order_level_dataset, load_olist_tables, modeling_frame
from src.geography import attach_order_geography


In [6]:
MAX_ROWS = 15_000  # Use None para la corrida completa.
config = ExperimentConfig(max_rows=MAX_ROWS, cv_splits=3, bootstrap_iterations=500)
config


ExperimentConfig(random_state=42, max_rows=15000, cv_splits=3, bootstrap_iterations=500, min_frequency=20, low_cardinality_max=30, high_cardinality_min=100)

In [7]:
tables = load_olist_tables(raw)
{name: df.shape for name, df in tables.items()}


{'orders': (99441, 8),
 'items': (112650, 7),
 'customers': (99441, 5),
 'products': (32951, 9),
 'payments': (103886, 5),
 'sellers': (3095, 4),
 'reviews': (99224, 7),
 'geolocation': (1000163, 5),
 'category_translation': (71, 2)}

In [8]:
orders = build_order_level_dataset(tables)
orders = attach_order_geography(orders, tables)
orders.shape


(99441, 37)

In [9]:
geo_cols = [
    'customer_lat', 'customer_lon',
    'mean_customer_seller_distance_km',
    'max_customer_seller_distance_km',
    'sellers_with_coordinates',
]
orders[geo_cols].describe()


,customer_lat,customer_lon,mean_customer_seller_distance_km,max_customer_seller_distance_km,sellers_with_coordinates
count,99163.000000,99163.000000,98177.000000,98177.000000,98666.000000
mean,-21.193324,-46.175700,601.592292,602.897490,1.008564
std,5.609607,4.058439,595.312790,595.794314,0.141151
min,-33.690972,-72.670987,0.000000,0.000000,0.000000
25%,-23.588569,-48.097766,186.518153,188.274065,1.000000
50%,-22.925419,-46.630866,433.769298,435.389942,1.000000
75%,-20.140685,-43.600962,798.024481,800.722774,1.000000
max,42.184003,-8.577855,8677.871551,8677.871551,5.000000


In [10]:
coverage = orders['mean_customer_seller_distance_km'].notna().mean()
print(f'Cobertura de distancia: {coverage:.2%}')


Cobertura de distancia: 98.73%


## 3. Regresión — días hasta la entrega

Corremos el pipeline completo (`tune_models`, `fit_best_and_test`, importancia por permutación sobre validación y comparación con PCA/UMAP). El ranking final se ordena por el score medio de validación cruzada; esa misma lógica es la que el baseline usó para elegir campeón.


In [11]:
from src.experiments import run_task


In [12]:
X_reg, y_reg, dates_reg = modeling_frame(orders, 'delivery_time_days')
X_reg.shape


(96476, 25)

In [13]:
reg = run_task('regression', X_reg, y_reg, dates_reg, config)


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/.venv/lib/python3.13/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [14]:
reg['ranking'][['model', 'train_MAE', 'validation_MAE', 'test_MAE', 'test_ci95_low', 'test_ci95_high']]


,model,train_MAE,validation_MAE,test_MAE,test_ci95_low,test_ci95_high
0,LinearSVR (SVM),4.996862,5.748942,4.611700,4.467956,4.744079
1,KNN (no parametrico),0.000001,6.114772,5.176658,5.017315,5.332214
2,MLP (red neuronal),5.219226,6.622786,5.543068,5.365551,5.697731
3,Ridge (parametrico),5.244072,6.700747,6.336246,6.175295,6.483794
4,Random Forest (ensamble),3.650578,6.845352,4.855147,4.702110,5.020480
5,Baseline trivial (mediana),6.384873,6.917828,5.043977,4.905983,5.182219


In [15]:
reg['feature_importance'].head(12)


,feature,importance_mean,importance_std
0,mean_customer_seller_distance_km,1.681618,0.067771
1,estimated_delivery_days,0.477133,0.015645
2,customer_state,0.349790,0.021276
3,product_category_name,0.104535,0.013242
4,max_customer_seller_distance_km,0.099520,0.026520
5,payment_type,0.033528,0.007781
6,customer_city,0.029433,0.010239
7,total_freight_value,0.015230,0.003634
8,total_price,0.011509,0.002425
9,item_count,0.011058,0.003353


In [16]:
reg['reductions']


,reduction,model,original_features,reduced_features,reduction_pct,MAE,RMSE,R2
0,PCA_95pct,LinearSVR (SVM),99,24,75.757576,4.688837,6.110324,-0.040468
1,PCA_95pct,KNN (no parametrico),99,24,75.757576,5.136992,6.602479,-0.214827
2,UMAP_10d,LinearSVR (SVM),99,10,89.898990,4.856228,6.171701,-0.061476
3,UMAP_10d,KNN (no parametrico),99,10,89.898990,5.738997,7.355848,-0.507877


## 4. Clasificación — probabilidad de entrega tardía

Misma lógica que la regresión pero sobre el objetivo binario `late_delivery`. La diferencia esperada aquí es menor: en el baseline el *shift* temporal entre validación y test dominaba el resultado, así que la distancia geográfica por sí sola no debería corregirlo.


In [17]:
X_cls, y_cls, dates_cls = modeling_frame(orders, 'late_delivery')
X_cls.shape


(96476, 25)

In [18]:
cls = run_task('classification', X_cls, y_cls, dates_cls, config)


In [19]:
cls['ranking'][['model', 'train_ROC_AUC', 'validation_ROC_AUC', 'test_ROC_AUC', 'test_ci95_low', 'test_ci95_high']]


,model,train_ROC_AUC,validation_ROC_AUC,test_ROC_AUC,test_ci95_low,test_ci95_high
0,Random Forest (ensamble),0.991036,0.714865,0.511994,0.470780,0.551532
1,KNN (no parametrico),1.000000,0.648879,0.514497,0.473990,0.553937
2,Logistica (parametrico),0.750103,0.629721,0.734426,0.703674,0.764691
3,MLP (red neuronal),0.550136,0.628286,0.388919,0.342291,0.435873
4,LinearSVC (SVM),0.750171,0.622194,0.729154,0.696278,0.759988
5,Baseline trivial (clase mayoritaria),0.500000,0.500000,0.500000,0.500000,0.500000


In [20]:
cls['feature_importance'].head(12)


,feature,importance_mean,importance_std
0,estimated_delivery_days,0.080663,0.002348
1,purchase_month,0.042837,0.002107
2,customer_state,0.031796,0.001730
3,total_freight_value,0.027582,0.002059
4,customer_lon,0.021884,0.001112
5,max_customer_seller_distance_km,0.019622,0.002652
6,mean_customer_seller_distance_km,0.019296,0.002514
7,customer_lat,0.015138,0.001113
8,product_category_name,0.015128,0.000569
9,customer_city,0.014110,0.000926


In [21]:
cls['reductions']


,reduction,model,original_features,reduced_features,reduction_pct,balanced_accuracy,precision,recall,F1,ROC_AUC
0,PCA_95pct,Random Forest (ensamble),99,24,75.757576,0.505812,0.666667,0.011976,0.023529,0.661895
1,PCA_95pct,KNN (no parametrico),99,24,75.757576,0.500000,0.000000,0.000000,0.000000,0.504518
2,UMAP_10d,Random Forest (ensamble),99,10,89.898990,0.475354,0.038877,0.107784,0.057143,0.393631
3,UMAP_10d,KNN (no parametrico),99,10,89.898990,0.500000,0.000000,0.000000,0.000000,0.453555


## 5. Comparación con el baseline

Cargamos el `summary.json` del experimento 01 y lo ponemos al lado de los números de esta corrida. La diferencia que importa es sobre test; el movimiento en validación no cuenta porque el *shift* temporal ya está documentado.


In [22]:
import json

baseline = json.loads((root / 'outputs/experimentos/01_baseline_pre_compra/summary.json').read_text())
baseline


{'sample_rows': 15000,
 'cv_splits': 3,
 'bootstrap_iterations': 500,
 'regression': {'champion': 'LinearSVR (SVM)',
  'test_primary': 4.693817039225055,
  'test_ci95': [4.535541577322184, 4.831942047499957]},
 'classification': {'champion': 'Random Forest (ensamble)',
  'test_primary': 0.4865708047371547,
  'test_ci95': [0.44577189846901355, 0.5277452676699849]}}

In [23]:
import pandas as pd

comparison = pd.DataFrame(
    [
        {
            'tarea': 'regresión (MAE días)',
            'baseline': baseline['regression']['test_primary'],
            'con distancia': float(reg['ranking'].iloc[0]['test_MAE']),
        },
        {
            'tarea': 'clasificación (ROC-AUC)',
            'baseline': baseline['classification']['test_primary'],
            'con distancia': float(cls['ranking'].iloc[0]['test_ROC_AUC']),
        },
    ]
)
comparison['delta'] = comparison['con distancia'] - comparison['baseline']
comparison


,tarea,baseline,con distancia,delta
0,regresión (MAE días),4.693817,4.611700,-0.082118
1,clasificación (ROC-AUC),0.486571,0.511994,0.025423


## 6. Persistencia de resultados

Guardamos las cuatro tablas por tarea y un `summary.json` que incluye la cobertura geográfica para dejar documentado el porcentaje de pedidos con coordenadas válidas. El `README.md` de la carpeta discute los hallazgos en prosa.


In [24]:
output = root / 'outputs' / 'experimentos' / '02_distancia_geografica'
output.mkdir(parents=True, exist_ok=True)

for prefix, result in [('regression', reg), ('classification', cls)]:
    for name in ['cv_results', 'ranking', 'feature_importance', 'reductions']:
        result[name].to_csv(output / f'{prefix}_{name}.csv', index=False)

print('Tablas guardadas en', output)


Tablas guardadas en /home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/outputs/experimentos/02_distancia_geografica


In [25]:
def _headline(result, primary_col):
    top = result['ranking'].iloc[0]
    return {
        'champion': top['model'],
        'test_primary': float(top[primary_col]),
        'test_ci95': [float(top['test_ci95_low']), float(top['test_ci95_high'])],
    }

summary = {
    'sample_rows': MAX_ROWS,
    'cv_splits': config.cv_splits,
    'bootstrap_iterations': config.bootstrap_iterations,
    'geo_coverage': float(coverage),
    'regression': _headline(reg, 'test_MAE'),
    'classification': _headline(cls, 'test_ROC_AUC'),
}
(output / 'summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)
summary


{'sample_rows': 15000,
 'cv_splits': 3,
 'bootstrap_iterations': 500,
 'geo_coverage': 0.9872889452036886,
 'regression': {'champion': 'LinearSVR (SVM)',
  'test_primary': 4.611699514771404,
  'test_ci95': [4.467956410917454, 4.744079044984438]},
 'classification': {'champion': 'Random Forest (ensamble)',
  'test_primary': 0.5119940140897168,
  'test_ci95': [0.4707802393631849, 0.5515316868556699]}}